In [ ]:
%pip install -q kagglehub
import kagglehub, os, shutil, random, glob

path = kagglehub.dataset_download("subirbiswas19/skin-disease-dataset")
train_candidates = glob.glob(os.path.join(path, '**', 'train_set'), recursive=True)
test_candidates = glob.glob(os.path.join(path, '**', 'test_set'), recursive=True)
if not train_candidates:
    train_candidates = glob.glob(os.path.join(path, '**', 'train'), recursive=True)
    test_candidates = glob.glob(os.path.join(path, '**', 'test'), recursive=True)
src_train = train_candidates[0]
src_test = test_candidates[0]

dst_base = "skin_small"
random.seed(42)
for src_dir, split in [(src_train, "train_set"), (src_test, "test_set")]:
    for cls in os.listdir(src_dir):
        s = os.path.join(src_dir, cls)
        if not os.path.isdir(s): continue
        d = os.path.join(dst_base, split, cls)
        os.makedirs(d, exist_ok=True)
        files = [f for f in os.listdir(s) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
        n = min(25, len(files))
        sel = random.sample(files, n)
        for f in sel:
            shutil.copy2(os.path.join(s, f), os.path.join(d, f))
print("Trimmed skin disease dataset ready")
TRAIN_DIR = os.path.join(dst_base, "train_set")
TEST_DIR = os.path.join(dst_base, "test_set")

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from tensorflow.keras import layers , models
from sklearn.utils.class_weight import compute_class_weight
import cv2 
from tensorflow.keras.applications import ResNet152V2

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir=TRAIN_DIR
test_dir=TEST_DIR

img_size = 224
batch_size = 32


data_gen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    fill_mode='nearest',
    validation_split=0.25
)



train_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    interpolation='bilinear',
    shuffle=True
)

valid_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    interpolation='bilinear',
    shuffle=False
)

print("Class indices:", train_gen.class_indices)
print("Class sample counts:", np.bincount(train_gen.classes))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

base_model = ResNet152V2(include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False  

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history=model.fit(
    train_gen,
    epochs=10,
    validation_data=valid_gen,
    class_weight=class_weights_dict
)

In [ ]:
model.save("ass3.h5")